In [ ]:
!pip -q install -U langchain langgraph langchain-openai pydantic ipython

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.9/107.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.9/471.9 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 54.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.7/625.7 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.7/508.7 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 5.9 MB/s eta 0:00:00


In [2]:
import os
import json
from getpass import getpass
from typing import TypedDict, List, Dict, Any, Literal

from IPython.display import display, Markdown
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

### Define structured schemas

In [3]:
class UserInput(BaseModel):
    user_goal: str
    recipient_name: str = ""
    recipient_role: str = ""
    sender_name: str = ""
    sender_role: str = ""
    relationship_context: str = ""
    message_context: str = ""
    key_points: List[str] = Field(default_factory=list)
    tone_preference: str = ""
    desired_length: Literal["short", "medium", "long"] = "medium"
    include_cta: bool = False
    edit_requests: List[str] = Field(default_factory=list)

class TaskClassification(BaseModel):
    intent: Literal[
        "job_application",
        "client_communication",
        "support_reply",
        "meeting_followup",
        "general_email"
    ]
    recommended_tone: str
    subject_needed: bool
    reply_mode: bool
    reason: str

In [4]:
class MissingDetails(BaseModel):
    missing_fields: List[str]
    placeholders: List[str]
    assumptions: List[str]

class TemplatePlan(BaseModel):
    template_name: str
    subject_style: str
    opening_style: str
    body_structure: List[str]
    closing_style: str

class DraftOutput(BaseModel):
    subject: str
    draft: str

class RefinedOutput(BaseModel):
    improved_version: str
    short_version: str
    formal_version: str
    cta_line: str

class FinalPackage(BaseModel):
    intent: str
    tone: str
    subject: str
    missing_fields: List[str]
    placeholders: List[str]
    assumptions: List[str]
    initial_draft: str
    improved_version: str
    short_version: str
    formal_version: str
    cta_line: str

### Create helper functions

In [5]:

def clean_text(value):
    return str(value).strip()

def clean_list(value):
    if isinstance(value, list):
        return [str(item).strip() for item in value if str(item).strip()]
    if isinstance(value, str):
        parts = [item.strip() for item in value.split("\n")]
        parts = [item for item in parts if item]
        if parts:
            return parts
    return []

def normalize_input_data(data):
    payload = UserInput(
        user_goal=clean_text(data.get("user_goal", "")),
        recipient_name=clean_text(data.get("recipient_name", "")),
        recipient_role=clean_text(data.get("recipient_role", "")),
        sender_name=clean_text(data.get("sender_name", "")),
        sender_role=clean_text(data.get("sender_role", "")),
        relationship_context=clean_text(data.get("relationship_context", "")),
        message_context=clean_text(data.get("message_context", "")),
        key_points=clean_list(data.get("key_points", [])),
        tone_preference=clean_text(data.get("tone_preference", "")),
        desired_length=clean_text(data.get("desired_length", "medium")) or "medium",
        include_cta=bool(data.get("include_cta", False)),
        edit_requests=clean_list(data.get("edit_requests", []))
    )
    return payload.model_dump()

In [6]:
def make_template(template_name, subject_style, opening_style, body_structure, closing_style):
    plan = TemplatePlan(
        template_name=template_name,
        subject_style=subject_style,
        opening_style=opening_style,
        body_structure=body_structure,
        closing_style=closing_style
    )
    return plan.model_dump()

In [7]:
def build_markdown_output(data):
    lines = []
    lines.append("# Professional Email Draft Agent Output")
    lines.append("")
    lines.append("## Detected Intent")
    lines.append(data["intent"])
    lines.append("")
    lines.append("## Recommended Tone")
    lines.append(data["tone"])
    lines.append("")
    lines.append("## Missing Details")
    if data["missing_fields"]:
        for item in data["missing_fields"]:
            lines.append(f"- {item}")
    else:
        lines.append("- No major missing details detected")
    lines.append("")
    lines.append("## Suggested Placeholders")
    if data["placeholders"]:
        for item in data["placeholders"]:
            lines.append(f"- {item}")
    else:
        lines.append("- No placeholder needed")
    lines.append("")
    lines.append("## Assumptions Used")
    if data["assumptions"]:
        for item in data["assumptions"]:
            lines.append(f"- {item}")
    else:
        lines.append("- No strong assumptions used")
    lines.append("")
    lines.append("## Subject")
    lines.append(data["subject"])
    lines.append("")
    lines.append("## Initial Draft")
    lines.append(data["initial_draft"])
    lines.append("")
    lines.append("## Improved Version")
    lines.append(data["improved_version"])
    lines.append("")
    lines.append("## Short Version")
    lines.append(data["short_version"])
    lines.append("")
    lines.append("## Formal Version")
    lines.append(data["formal_version"])
    lines.append("")
    lines.append("## CTA Line")
    lines.append(data["cta_line"] if data["cta_line"] else "No CTA line added")
    return "\n".join(lines)

### Define workflow state

In [ ]:
class EmailState(TypedDict):
    raw_input: Dict[str, Any]
    input_data: Dict[str, Any]
    classification: Dict[str, Any]
    missing: Dict[str, Any]
    template: Dict[str, Any]
    initial_draft: Dict[str, Any]
    refined: Dict[str, Any]
    final_output: Dict[str, Any]
    report_markdown: str